# Module 08 · Unit 02
# Cryptographic Primitives

| | |
|---|---|
| **Estimated time** | 65–75 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Cryptographic Structure \[CS\] |
| **Refresher** | Module 08 · Unit 01 — Divisibility and Modular Arithmetic |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Define Euler's totient function $\phi(n)$ and compute it for prime products
2. State **Euler's theorem** and derive Fermat's Little Theorem as a special case
3. Explain **RSA key generation** step by step from first principles
4. Prove that RSA decryption recovers the original plaintext using Euler's theorem
5. Explain why breaking RSA requires factoring $n$ — the hardness assumption
6. Describe **Diffie-Hellman key exchange** and the discrete logarithm problem
7. Connect these primitives to the TLS connections securing every AI API deployment


## 🔣 Symbol Primer

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $\phi(n)$ | Euler's totient | "phi of $n$" | Number of integers in $\{1,\ldots,n\}$ coprime to $n$ |
| $\mathbb{Z}_n^*$ | Units group | "Z sub $n$ star" | $\{a \in \{1,\ldots,n-1\} \mid \gcd(a,n)=1\}$ — has $\phi(n)$ elements |
| $(e, n)$ | RSA public key | "public key" | Encryption exponent $e$ and modulus $n = pq$ |
| $(d, n)$ | RSA private key | "private key" | Decryption exponent $d = e^{-1} \bmod \phi(n)$ |
| $c = m^e \bmod n$ | RSA encryption | "ciphertext" | Plaintext $m$ encrypted with public key |
| $m = c^d \bmod n$ | RSA decryption | "plaintext" | Ciphertext $c$ decrypted with private key |
| $g$ | Generator | "generator" | A primitive root mod $p$ — Diffie-Hellman base |
| $g^a \bmod p$ | DH public value | "DH public" | Alice's public Diffie-Hellman value |

---


## 1 · Euler's Totient Function

**Euler's totient function** $\phi(n)$ counts the integers in $\{1, 2, \ldots, n\}$
that are coprime to $n$:

$$\phi(n) = |\{a \in \{1,\ldots,n\} \mid \gcd(a,n) = 1\}| = |\mathbb{Z}_n^*|$$

### Key Values

For a prime $p$:
$$\phi(p) = p - 1$$

Every integer from 1 to $p-1$ is coprime to a prime.

For a product of two distinct primes $n = pq$:
$$\phi(pq) = (p-1)(q-1)$$

The integers NOT coprime to $pq$ are exactly the multiples of $p$ or $q$ in
$\{1,\ldots,pq\}$. There are $q$ multiples of $p$ and $p$ multiples of $q$, with
1 overlap ($pq$ itself): $p + q - 1$ non-coprime integers, so $\phi(pq) = pq - p - q + 1 = (p-1)(q-1)$.

This formula is the key to RSA. Computing $\phi(n) = (p-1)(q-1)$ is easy when you
know $p$ and $q$, but hard when you only know $n = pq$ without knowing the factors.


## 2 · Euler's Theorem

**Theorem (Euler).** For any integer $a$ with $\gcd(a, n) = 1$:

$$a^{\phi(n)} \equiv 1 \pmod{n}$$

This directly generalises Fermat's Little Theorem: when $n = p$ is prime,
$\phi(p) = p-1$, and Euler's theorem becomes $a^{p-1} \equiv 1 \pmod{p}$.

### Proof Sketch

The proof mirrors the Fermat proof from Unit 01. Consider the units group
$\mathbb{Z}_n^*$ with $\phi(n)$ elements $\{u_1, u_2, \ldots, u_{\phi(n)}\}$.
Since $\gcd(a, n) = 1$, multiplying each $u_i$ by $a$ permutes $\mathbb{Z}_n^*$.
Therefore:

$$\prod_{i} (au_i) \equiv \prod_{i} u_i \pmod{n}$$
$$a^{\phi(n)} \prod_{i} u_i \equiv \prod_{i} u_i \pmod{n}$$

Since $\gcd(\prod u_i, n) = 1$, dividing gives $a^{\phi(n)} \equiv 1 \pmod{n}$. $\square$

### The RSA Consequence

From Euler's theorem: $a^{\phi(n)} \equiv 1$, so $a^{k \cdot \phi(n)} \equiv 1$
for any positive integer $k$. Therefore:

$$a^{k \cdot \phi(n) + 1} \equiv a \pmod{n}$$

RSA decryption works because the product $ed$ satisfies $ed \equiv 1 \pmod{\phi(n)}$,
meaning $ed = k \cdot \phi(n) + 1$ for some integer $k$. So:

$$(m^e)^d = m^{ed} = m^{k\phi(n)+1} \equiv m \pmod{n} \qquad \square$$

This is the RSA correctness proof.


## 3 · RSA Key Generation — Step by Step

### The Procedure

**Step 1.** Choose two large distinct primes $p$ and $q$.

**Step 2.** Compute the modulus $n = pq$.

**Step 3.** Compute $\phi(n) = (p-1)(q-1)$.

**Step 4.** Choose a public exponent $e$ with $1 < e < \phi(n)$ and $\gcd(e, \phi(n)) = 1$.
Common choices: $e = 65537 = 2^{16} + 1$ (used in most deployments).

**Step 5.** Compute the private exponent $d = e^{-1} \bmod \phi(n)$ using the
Extended Euclidean Algorithm.

**Public key:** $(e, n)$ — published openly.
**Private key:** $(d, n)$ — kept secret. **Also keep $p$, $q$, $\phi(n)$ secret.**

### RSA Encryption and Decryption

Given message $m$ with $0 \leq m < n$:

**Encryption:** $c = m^e \bmod n$

**Decryption:** $m = c^d \bmod n$

Correctness: $c^d = (m^e)^d = m^{ed} \equiv m \pmod{n}$ by Euler's theorem (Section 2).

### Security Assumption

The security of RSA rests on the **integer factorisation problem**:
given $n = pq$, compute $p$ and $q$. If an attacker can factor $n$, they can
compute $\phi(n) = (p-1)(q-1)$ and then $d = e^{-1} \bmod \phi(n)$ — recovering
the private key.

No polynomial-time factorisation algorithm is known. The best known classical
algorithm (General Number Field Sieve) runs in sub-exponential time — much faster
than brute force, but still infeasible for 2048-bit primes.

**Quantum threat.** Shor's algorithm runs in polynomial time on a quantum computer
and would break RSA. Post-quantum cryptography (lattice-based schemes, hash-based
signatures) is actively being standardised to replace RSA in a post-quantum world.


## 4 · Diffie-Hellman Key Exchange

RSA solves the problem of encrypting a message with a public key. But it doesn't
solve the problem of two parties who have never met establishing a shared secret
over a public channel — which is what every TLS connection does.

**Diffie-Hellman** solves this. Both parties agree publicly on:
- A large prime $p$
- A generator $g$ (a primitive root mod $p$: $g^1, g^2, \ldots, g^{p-1}$ covers all of $\mathbb{Z}_p^*$)

**Key exchange:**

1. **Alice** chooses private $a$, computes $A = g^a \bmod p$, sends $A$ to Bob.
2. **Bob** chooses private $b$, computes $B = g^b \bmod p$, sends $B$ to Alice.
3. **Alice** computes $K = B^a \bmod p = g^{ab} \bmod p$.
4. **Bob** computes $K = A^b \bmod p = g^{ab} \bmod p$.
5. Both have the same shared secret $K = g^{ab} \bmod p$.

An eavesdropper sees $p$, $g$, $A = g^a$, and $B = g^b$ — but to compute $K = g^{ab}$,
they would need to find $a$ from $A = g^a$. This is the **discrete logarithm problem**:

$$\text{Given } g, A, p,\; \text{find } a \text{ such that } g^a \equiv A \pmod{p}$$

No efficient classical algorithm is known for this problem when $p$ is large.

### Forward Secrecy

Modern TLS uses **Ephemeral Diffie-Hellman (DHE)** or **Elliptic Curve Diffie-Hellman (ECDHE)**
where new random values $a$ and $b$ are chosen for every connection. Even if the server's
long-term private key is compromised later, past session keys cannot be recovered —
each session's $g^{ab}$ is gone. This property is called **perfect forward secrecy (PFS)**.


## 5 · TLS and AI System Security

Every API call an AI model makes — to retrieve data, call tools, authenticate
users — travels over TLS. The handshake that secures that connection uses exactly
the primitives from this module.

### A Simplified TLS 1.3 Handshake

1. **Client Hello:** client sends supported cipher suites and a random nonce
2. **Server Hello:** server selects cipher suite (e.g., TLS_AES_256_GCM_SHA384 with ECDHE)
3. **Key exchange:** ECDHE parameters are exchanged; both sides compute the shared secret $K$
4. **Certificate:** server proves identity via RSA or ECDSA signature on its certificate
5. **Finished:** both sides derive session keys from $K$ and confirm the handshake
6. **Application data:** all subsequent data is encrypted with AES-GCM using the session keys

### The Math Behind Each Step

| TLS step | Mathematical primitive | Module |
|---|---|---|
| ECDHE key exchange | Discrete logarithm on elliptic curves | Module 08 (DH analogue) |
| RSA/ECDSA certificate | RSA signature verification | Module 08 (RSA) |
| AES-GCM encryption | Bijective permutation on bit blocks | Module 03 (bijections) |
| SHA-384 integrity | Hash function collision resistance | Module 03 + Module 07 (Pigeonhole) |
| Session key derivation | Modular arithmetic + pseudorandom functions | Module 08 |

**The AI security connection.** An agentic AI system calling an MCP server, an LLM
API, or a database — every one of those connections is secured by exactly this
chain of mathematics. Understanding TLS means understanding why intercepting an AI
agent's traffic requires breaking either the discrete logarithm problem or factoring
a 2048-bit RSA modulus — and why neither is currently feasible.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[CS\]

| Cryptographic concept | Mathematical foundation | Security assumption |
|---|---|---|
| **RSA encryption** | $c = m^e \bmod n$ — modular exponentiation | Integer factorisation is hard |
| **RSA decryption** | $m = c^d \bmod n$, correct by Euler's theorem | Private key derivation requires factoring $n$ |
| **RSA key generation** | $d = e^{-1} \bmod \phi(n)$ — Extended Euclidean | $\phi(n)$ is secret without knowing $p$, $q$ |
| **Diffie-Hellman** | $K = g^{ab} \bmod p$ — shared secret | Discrete logarithm is hard |
| **Forward secrecy** | Fresh DH parameters per session | Compromising one session doesn't expose others |
| **TLS AES encryption** | Block cipher permutation | Pseudorandom permutation hardness |
| **Quantum threat** | Shor's algorithm breaks factoring and DLog in poly time | Post-quantum migration needed |

**The chain of trust.** The security of every AI API call ultimately rests on:
1. The hardness of integer factorisation (RSA) or discrete logarithm (DH/ECDH)
2. The pseudorandomness of AES (Module 03 bijections + algebraic structure)
3. The collision resistance of SHA-384/SHA-256 (Module 07 birthday bound)

These are not separate concerns — they are the interlocking mathematical guarantees
that together make "secure communication" mean something precise.

---


## Python: Visualization & Verification

**1 · RSA from Scratch** — implement RSA key generation, encryption, and
decryption from first principles using only the tools from this module; verify the
correctness proof computationally.

**2 · Diffie-Hellman Key Exchange** — simulate a DH key exchange between Alice
and Bob; visualise what an eavesdropper sees vs what the parties share; verify
that both parties arrive at the same secret.

**3 · Factoring vs Security** — plot the growth of factoring difficulty as a
function of RSA key size; compare the timeline for classical and quantum attacks;
show why key sizes have grown over the decades.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from math import gcd, log2, log10, isqrt
import random

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

# Helper functions from Unit 01
def extended_gcd(a, b):
    if b == 0: return a, 1, 0
    g, s, t = extended_gcd(b, a % b)
    return g, t, s - (a // b) * t

def mod_inverse(a, n):
    g, s, _ = extended_gcd(a, n)
    return s % n if g == 1 else None

def euler_totient(n, p, q):
    """Totient for n = p*q."""
    return (p - 1) * (q - 1)

print('Setup complete.')


### 1 · RSA from Scratch

We implement complete RSA key generation, encryption, and decryption using only
the primitives from Unit 01. We use small primes (pedagogically clear) but note
where real implementations differ (2048-bit primes, OAEP padding, side-channel
protections). We then verify the correctness proof: $c^d \equiv m \pmod{n}$.


In [ ]:
# ── 1 · RSA from Scratch ──────────────────────────────────────────────────────

class RSA:
    """Toy RSA implementation for pedagogical clarity. NOT FOR PRODUCTION USE."""

    def __init__(self, p, q, e=None):
        assert p != q, "p and q must be distinct primes"
        self.p = p
        self.q = q
        self.n = p * q
        self.phi_n = euler_totient(self.n, p, q)

        # Choose public exponent e
        if e is None:
            # Default: find a small e coprime to phi(n)
            for candidate in [3, 5, 17, 257, 65537]:
                if gcd(candidate, self.phi_n) == 1:
                    e = candidate
                    break
        assert gcd(e, self.phi_n) == 1, "e must be coprime to phi(n)"
        self.e = e
        self.d = mod_inverse(e, self.phi_n)

    def encrypt(self, m):
        assert 0 < m < self.n, f"Message must be in (0, {self.n})"
        return pow(m, self.e, self.n)

    def decrypt(self, c):
        return pow(c, self.d, self.n)

    def summary(self):
        print(f"RSA Parameters:")
        print(f"  p = {self.p},  q = {self.q}")
        print(f"  n = p×q = {self.n}")
        print(f"  φ(n) = (p-1)(q-1) = {self.phi_n}")
        print(f"  e = {self.e}  [public exponent — gcd(e,φ(n)) = {gcd(self.e, self.phi_n)}]")
        print(f"  d = e⁻¹ mod φ(n) = {self.d}  [private exponent]")
        print(f"  Verify: e×d mod φ(n) = {(self.e * self.d) % self.phi_n}  ✓" if
              (self.e * self.d) % self.phi_n == 1 else "  VERIFY FAILED")
        print(f"  Public key:  (e={self.e}, n={self.n})")
        print(f"  Private key: (d={self.d}, n={self.n})")

# Use the classic textbook example: p=61, q=53 (giving n=3233)
rsa = RSA(p=61, q=53, e=17)
rsa.summary()

# Encrypt and decrypt several messages
print(f"
Encrypt/Decrypt verification:")
print(f"{'m (plaintext)':>16} {'c = m^e mod n':>18} {'m\'= c^d mod n':>18} {'Match':>8}")
print("─" * 65)
for m in [65, 123, 42, 999, 2000, 3000]:
    if 0 < m < rsa.n:
        c  = rsa.encrypt(m)
        m2 = rsa.decrypt(c)
        ok = '✓' if m == m2 else '✗'
        print(f"  {m:>14}  {c:>18}  {m2:>18}  {ok:>7}")

# ── Correctness proof visualised ───────────────────────────────────────────────
print(f"
Correctness proof trace for m=65:")
m_demo = 65
c_demo = rsa.encrypt(m_demo)
ed     = rsa.e * rsa.d
k      = (ed - 1) // rsa.phi_n
print(f"  e×d = {rsa.e} × {rsa.d} = {ed}")
print(f"  e×d = k×φ(n) + 1 where k = {k}")
print(f"  m^(e×d) = m^(k×φ(n)+1)")
print(f"  By Euler's theorem: m^φ(n) ≡ 1 (mod n)")
print(f"  → m^(k×φ(n)) ≡ 1^k = 1 (mod n)")
print(f"  → m^(k×φ(n)+1) ≡ m (mod n)")
print(f"  Verify: {m_demo}^{ed} mod {rsa.n} = {pow(m_demo, ed, rsa.n)}  (= m = {m_demo}) ✓")


**What do you see?**

Every message encrypts and then decrypts back to itself — the bijection property
from Module 03 demonstrated concretely. The correctness proof trace shows the
algebra: $e \times d = k \cdot \phi(n) + 1$ for some integer $k$, and Euler's
theorem guarantees $m^{\phi(n)} \equiv 1$, so $m^{ed} = m^{k\phi(n)+1} \equiv m$.

Notice that $e = 17$ and $d = 2753$: the public exponent is small (fast encryption),
but the private exponent is large (slower decryption — the asymmetry in RSA performance).
Real implementations use the Chinese Remainder Theorem to speed up decryption by
about 4× by working in $\mathbb{Z}_p$ and $\mathbb{Z}_q$ separately.

**The Module 00 chain is complete.** Back in Unit 01 of Module 00, proof by
contradiction was promised to prove uniqueness of modular inverses for RSA.
Unit 01 of this module delivered that proof. This unit delivers the full RSA
system that depends on it.


### 2 · Diffie-Hellman Key Exchange

We simulate a complete DH key exchange between Alice and Bob over a public channel,
visualising exactly what an eavesdropper Eve can observe — and what she cannot compute
without solving the discrete logarithm problem.


In [ ]:
# ── 2 · Diffie-Hellman Key Exchange ─────────────────────────────────────────
random.seed(42)

# Public DH parameters (small for demonstration)
p = 2357    # a prime
g = 2       # a generator of Z_p*

# Private keys (kept secret)
a_private = random.randint(2, p-2)   # Alice's private key
b_private = random.randint(2, p-2)   # Bob's private key

# Public values (sent over open channel)
A_public = pow(g, a_private, p)    # Alice sends: g^a mod p
B_public = pow(g, b_private, p)    # Bob sends:   g^b mod p

# Shared secrets (computed independently)
K_alice = pow(B_public, a_private, p)   # Alice: B^a = (g^b)^a = g^ab
K_bob   = pow(A_public, b_private, p)   # Bob:   A^b = (g^a)^b = g^ab

print("Diffie-Hellman Key Exchange Simulation")
print("=" * 55)
print(f"
Public parameters (known to everyone including Eve):")
print(f"  Prime p = {p}")
print(f"  Generator g = {g}")
print(f"
Alice's private key: a = {a_private}  [SECRET — Eve cannot see this]")
print(f"Bob's private key:   b = {b_private}  [SECRET — Eve cannot see this]")
print(f"
Public channel (Eve observes all of this):")
print(f"  Alice → Bob: A = g^a mod p = {g}^{a_private} mod {p} = {A_public}")
print(f"  Bob → Alice: B = g^b mod p = {g}^{b_private} mod {p} = {B_public}")
print(f"
Shared secret computation:")
print(f"  Alice: K = B^a mod p = {B_public}^{a_private} mod {p} = {K_alice}")
print(f"  Bob:   K = A^b mod p = {A_public}^{b_private} mod {p} = {K_bob}")
print(f"
Shared secrets match: {K_alice == K_bob} ✓" if K_alice == K_bob else "
MISMATCH ✗")
print(f"
Eve knows: p={p}, g={g}, A={A_public}, B={B_public}")
print(f"To find K, Eve must compute a from A=g^a mod p → discrete logarithm problem")

# ── Visualise what Eve sees ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 10); ax.set_ylim(0, 6)
ax.axis('off'); ax.set_facecolor('white'); fig.patch.set_facecolor('white')

def box(ax, x, y, text, color, w=1.6, h=0.7):
    rect = mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h,
        boxstyle='round,pad=0.1', facecolor=color, edgecolor='white', lw=2)
    ax.add_patch(rect)
    for i, line in enumerate(text.split('
')):
        ax.text(x, y + 0.12 - i*0.25, line, ha='center', va='center',
                fontsize=8, fontweight='bold', color='white')

def arrow(ax, x0, y0, x1, y1, label='', color=TS_GRAY):
    ax.annotate('', xy=(x1,y1), xytext=(x0,y0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    if label:
        ax.text((x0+x1)/2, (y0+y1)/2+0.2, label, ha='center',
                fontsize=8, color=color, style='italic')

# Parties
box(ax, 1.5, 4.5, 'ALICE
secret: a', TS_BLUE)
box(ax, 8.5, 4.5, 'BOB
secret: b', TS_AMBER)
box(ax, 5.0, 1.0, 'EVE
sees: p,g,A,B
CANNOT compute K', TS_RED, w=2.5)

# Public parameters
ax.text(5, 5.5, f'Public: p={p}, g={g}', ha='center', fontsize=9,
        color=TS_GRAY, style='italic')

# Arrows for public values
arrow(ax, 2.3, 4.3, 7.7, 4.3,
      f'A = g^a mod p = {A_public}', TS_BLUE)
arrow(ax, 7.7, 3.8, 2.3, 3.8,
      f'B = g^b mod p = {B_public}', TS_AMBER)

# Shared secret computation
box(ax, 1.5, 2.5, f'K = B^a mod p
= {K_alice}', TS_GREEN)
box(ax, 8.5, 2.5, f'K = A^b mod p
= {K_bob}', TS_GREEN)

ax.text(5, 3.0, '= g^{ab} mod p', ha='center', fontsize=11,
        color=TS_GREEN, fontweight='bold')
ax.text(5, 2.5, f'(same shared secret K = {K_alice})', ha='center',
        fontsize=9, color=TS_GREEN)

# Eve's observation
ax.annotate('', xy=(4.0,1.4), xytext=(5.0,3.8),
            arrowprops=dict(arrowstyle='->', color=TS_RED, lw=1.5, linestyle='dashed'))
ax.text(3.8, 2.6, 'Observes
but cannot
compute K', ha='center',
        fontsize=8, color=TS_RED, style='italic')

ax.set_title('Diffie-Hellman Key Exchange
'
             'Alice and Bob establish a shared secret over a public channel',
             fontsize=11, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()


**What do you see?**

The diagram shows exactly what each party knows at each stage. Eve observes
everything on the public channel — the prime $p$, generator $g$, and both public
values $A$ and $B$ — but cannot compute the shared secret $K = g^{ab} \bmod p$
without knowing either $a$ or $b$.

To find $a$ from $A = g^a \bmod p$, Eve must solve the discrete logarithm:
given $g$, $A$, and $p$, find the exponent. For the small $p = 2357$ used here,
this is trivial. For $p$ with 3000+ bits (modern deployments), the best known
algorithms are sub-exponential but still infeasible.

**The forward secrecy insight.** Because $a$ and $b$ are generated freshly for
each session and discarded after the handshake, Eve's observation of $A$ and $B$
gives her no advantage for any past or future session. Even if she later compromises
Alice's long-term key, she cannot recover this session's $K$.


### 3 · Key Size vs Security — The Arms Race

RSA key sizes have grown steadily as computers have become faster. We plot the
estimated security level (bits of work to break) against key size for both
classical (GNFS) and quantum (Shor's) attacks, showing why 2048-bit keys are
the current minimum and why post-quantum migration is urgent.


In [ ]:
# ── 3 · Key Size vs Security Level ───────────────────────────────────────────

# GNFS complexity approximation for RSA n-bit modulus:
# log2(operations) ≈ 1.923 * (n * (log2(n))^2)^(1/3)
def gnfs_security(n_bits):
    """Approximate log2 of GNFS operation count for n-bit RSA key."""
    import math
    log2_n = n_bits
    return 1.923 * (n_bits * log2_n**2)**(1/3)

# Shor's algorithm: polynomial time → roughly O(n^3) quantum gate operations
def shor_security(n_bits):
    """Log2 of Shor's algorithm gate count (approximate)."""
    return 3 * log2(n_bits)

# Historical and recommended key sizes
key_sizes_rsa  = list(range(512, 4097, 64))
classical_sec  = [gnfs_security(n) for n in key_sizes_rsa]
quantum_sec    = [shor_security(n) for n in key_sizes_rsa]

milestones = [
    (512,  '512-bit
(broken 1999)', TS_RED),
    (768,  '768-bit
(broken 2009)', TS_RED),
    (1024, '1024-bit
(deprecated)', TS_AMBER),
    (2048, '2048-bit
(current min)', TS_GREEN),
    (4096, '4096-bit
(high security)', TS_BLUE),
]

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(key_sizes_rsa, classical_sec, color=TS_BLUE, lw=2.5,
        label='Classical attack (GNFS)')
ax.plot(key_sizes_rsa, quantum_sec,   color=TS_RED, lw=2.5, linestyle='--',
        label="Quantum attack (Shor's algorithm)")

# Security level reference lines
for level, label, color in [(56, 'DES (broken)', TS_RED),
                              (80, 'Minimum (deprecated)', TS_AMBER),
                              (112, '3DES equivalent', TS_AMBER),
                              (128, 'AES-128 equivalent
(target)', TS_GREEN)]:
    ax.axhline(level, color=color, lw=1.2, linestyle=':', alpha=0.7, label=label)

# Milestone annotations
for size, label, color in milestones:
    sec_c = gnfs_security(size)
    ax.scatter([size], [sec_c], color=color, s=120, zorder=5)
    ax.text(size, sec_c + 2, label, ha='center', fontsize=7, color=color)

ax.set_xlabel('RSA Key Size (bits)')
ax.set_ylabel('Security level (log₂ of operations to break)')
ax.set_title('RSA Key Size vs Security — Classical vs Quantum Attacks
'
             "Dotted lines = security level targets | Shor's makes ANY key size insecure",
             pad=10, fontsize=11)
ax.legend(fontsize=8, loc='upper left', ncol=2)
ax.set_ylim(0, 160)
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()

# Print key size recommendations
print("Current RSA key size guidance:")
for size in [1024, 2048, 3072, 4096]:
    classical = gnfs_security(size)
    quantum   = shor_security(size)
    status = ('BROKEN (deprecated)' if classical < 80 else
              'Minimum acceptable' if classical < 112 else
              'Recommended' if classical < 140 else 'High security')
    print(f"  {size:>5}-bit: classical={classical:.0f}-bit security, "
          f"quantum={quantum:.1f}-bit → {status}")

print(f"
Post-quantum note:")
print(f"  Shor's algorithm reduces ANY RSA key to ~{shor_security(2048):.0f} quantum bits.")
print(f"  A 2048-bit key requires ~4000 logical qubits to break with Shor's.")
print(f"  Current quantum computers: ~1000 noisy physical qubits → RSA still safe.")
print(f"  Timeline to cryptographically relevant quantum computer: disputed, est. 10-20 years.")
print(f"  NIST post-quantum standards (ML-KEM, ML-DSA, SLH-DSA) finalised 2024.")


**What do you see?**

The blue GNFS curve rises with key size — larger classical keys provide more
security. The red Shor's curve is nearly flat and very low — Shor's algorithm
is polynomial, so quantum computers break any RSA key with roughly the same
(small) number of quantum gates regardless of key size.

The vertical milestone markers show the history: 512-bit RSA was broken in 1999,
768-bit in 2009. The 2048-bit standard provides about 112 bits of classical
security — enough for now, but under continued pressure.

The post-quantum note at the bottom grounds the theory in current reality: 2024
saw NIST finalise the first post-quantum standards. The migration from RSA to
lattice-based cryptography is not a future concern — it is an active engineering
project happening in production systems right now.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. RSA WITH CRT OPTIMISATION
#    RSA decryption can be ~4× faster using the Chinese Remainder Theorem.
#    Instead of computing c^d mod n (large), compute:
#      m_p = c^(d mod p-1) mod p
#      m_q = c^(d mod q-1) mod q
#    Then use CRT to combine m_p and m_q into m mod n.
#    Implement crt_decrypt(c, d, p, q, n) and verify it gives the same result.
#    Time both methods for several messages and compute the speedup.
#
# 2. DISCRETE LOGARITHM — BABY-STEP GIANT-STEP
#    The Baby-Step Giant-Step algorithm solves the DLog in O(√p) time:
#      Goal: find x such that g^x ≡ A (mod p)
#      Baby steps: precompute {g^j mod p : j = 0, ..., √p}
#      Giant steps: compute A × (g^(-√p))^i mod p for i = 0, ..., √p
#                   until a match is found in the baby-step table
#    Implement and verify it finds the correct discrete log for small p.
#    Why does this NOT break DH for large p? (What is √p for a 3000-bit p?)
#
# 3. RSA SIGNATURE
#    RSA can also be used for digital signatures (prove identity without a key exchange):
#      Sign:   s = m^d mod n  (private key operation)
#      Verify: m' = s^e mod n, check m' == m  (public key operation)
#    Implement sign(m, d, n) and verify(s, e, n).
#    Why does this work? (Hint: same correctness proof as encryption.)
#    Why is the SIGNER using the PRIVATE key? (Roles are reversed from encryption.)

# Your work here:


---

## Summary

| Concept | Definition | Cryptographic role |
|---|---|---|
| $\phi(n)$ | $\|\{a \leq n : \gcd(a,n)=1\}\|$ | RSA parameter — hard to compute without factors |
| $\phi(pq)=(p-1)(q-1)$ | Totient of RSA modulus | The key to RSA's security |
| Euler's theorem | $a^{\phi(n)} \equiv 1 \pmod{n}$ | RSA correctness proof |
| RSA public key $(e,n)$ | $e$ coprime to $\phi(n)$ | Encrypt with $m^e \bmod n$ |
| RSA private key $(d,n)$ | $d = e^{-1} \bmod \phi(n)$ | Decrypt with $c^d \bmod n$ |
| RSA security | Integer factorisation hardness | Factoring $n = pq$ recovers $\phi(n)$ and $d$ |
| Diffie-Hellman | $K = g^{ab} \bmod p$ | Shared secret without pre-shared key |
| Discrete log problem | Given $g^a \bmod p$, find $a$ | DH security assumption |
| Forward secrecy | Fresh DH params per session | Past sessions safe even if key compromised |
| Quantum threat | Shor's algorithm | Post-quantum migration needed |

---

## Module 08 Complete

| Unit | Content | Payoff |
|---|---|---|
| **01** | Divisibility, GCD, modular arithmetic, inverses, Fermat's LT | The arithmetic foundation of all deployed cryptography |
| **02** | Euler's theorem, RSA, Diffie-Hellman, TLS, quantum threat | Full derivation of why every AI API connection is secure |

---

## Up Next

**Module 09 — Automata & Formal Languages**

From numbers to strings. Automata theory gives us the formal language for
describing systems that change state in response to inputs — protocols,
parsers, and regex engines. The MCP handshake is a finite automaton;
a dangerous regex pattern is an NFA with exponential backtracking; a grammar
is a generative system for structured inputs. Module 09 brings the formal tools
to protocol verification and input validation.

$\rightarrow$ `modules/module-09/unit-01-finite-automata.ipynb`
